# Fase 3 · M06: Alerta Temprana (Casos D y D_strict)

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M06 — Alerta Temprana |

---

## 🎯 Qué hace

Construye los datasets D y D_strict con restricción de alerta temprana real: solo variables disponibles en el momento de intervención (primer año). Incluye baselines rápidos de validación.

## 📋 Requisitos

- `data/automl/df_exp_automl_C.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/automl/df_exp_automl_D.parquet` | Caso D — 22 features, sin leakage |
| `data/automl/df_exp_automl_D_strict.parquet` | Caso D_strict — 19 features, más restrictivo |
| `data/automl/quick_baseline_casoD.parquet` | Baseline rápido de validación |
| `docs/html/fase3/m06_alerta_temprana.html` | Informe HTML |

## 🔄 Flujo

```
df_exp_automl_C.parquet
    ↓ Filtrado variables disponibles en 1er año
    ↓ Caso D (22 features) + Caso D_strict (19 features)
    ↓ Baselines de validación (RF, LogReg)
    → df_exp_automl_D.parquet + df_exp_automl_D_strict.parquet
```

## ➡️ Siguiente

`f3_m07_validacion.ipynb` — validación completa con feature importance


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))

from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import generar_kpis_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

RUTA_FASE3_HTML = RUTA_HTML / 'fase3'
crear_directorios([RUTA_AUTOML, RUTA_FASE3_HTML])
fmt = formato_numero_es

print('✓ Configuración completada')
info_entorno()

✓ Directorios verificados: 2
✓ Configuración completada
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_pro

In [2]:
# ============================================================================
# CELDA 2: CARGAR CASO C Y CREAR CASO D + D_STRICT
# ============================================================================

print('='*70)
print('F3-M06: CASOS D y D_STRICT — ALERTA TEMPRANA REAL')
print('='*70)

# Cargar Caso C
df_automl_c = pd.read_parquet(RUTA_AUTOML / 'df_exp_automl_C.parquet')
print(f'\n📥 Caso C cargado: {df_automl_c.shape[0]:,} × {df_automl_c.shape[1]} cols')
print(f'   Columnas: {list(df_automl_c.columns)}')


# ╔════════════════════════════════════════════════════════════════╗
# ║  CASO D — Script 1 del director del proyecto                  ║
# ║  Elimina 11 variables: las que acumulan historia completa     ║
# ╚════════════════════════════════════════════════════════════════╝

COLS_CASO_D = [
    'indicador_sin_notas',      # Sin notas → ya abandonó
    'nota_ultimo_anio',         # Nota 0 en último año → abandonó
    'cred_superados_total',     # Acumula toda la historia
    'cred_matriculados_total',  # Acumula toda la historia
    'cred_superados_anio_medio',# Necesita años totales
    'tasa_rendimiento',         # Acumula todo
    'ratio_avance',             # Foto del final
    'velocidad_creditos',       # Temporal
    'progreso_relativo',        # Foto del final
    'media_global',             # Usa notas de años posteriores
    'cred_titulacion',          # Constante, no aporta
]

# ╔════════════════════════════════════════════════════════════════╗
# ║  CASO D_STRICT — Script 2 del director (más agresivo)         ║
# ║  Añade estabilidad_academica y mejora_notas                   ║
# ╚════════════════════════════════════════════════════════════════╝

COLS_CASO_D_STRICT = COLS_CASO_D + [
    'estabilidad_academica',    # Calculada con toda la trayectoria
    'mejora_notas',             # Si usa notas posteriores
]

# --- Crear Caso D ---
print(f'\n--- CASO D (elimina {len(COLS_CASO_D)} variables) ---')
cols_d = [c for c in COLS_CASO_D if c in df_automl_c.columns]
df_automl_d = df_automl_c.drop(columns=cols_d)
print(f'   Eliminadas: {cols_d}')
print(f'   AutoML: {df_automl_d.shape[1]} cols ({df_automl_d.shape[1]-1} features + abandono)')
print(f'   Columnas: {list(df_automl_d.columns)}')

# --- Crear Caso D_strict ---
print(f'\n--- CASO D_STRICT (elimina {len(COLS_CASO_D_STRICT)} variables) ---')
cols_ds = [c for c in COLS_CASO_D_STRICT if c in df_automl_c.columns]
df_automl_ds = df_automl_c.drop(columns=cols_ds)
print(f'   Eliminadas: {cols_ds}')
print(f'   AutoML: {df_automl_ds.shape[1]} cols ({df_automl_ds.shape[1]-1} features + abandono)')
print(f'   Columnas: {list(df_automl_ds.columns)}')

# Resumen
print(f'\n📊 Resumen:')
print(f'   Caso C:        {df_automl_c.shape[1]} cols')
print(f'   Caso D:        {df_automl_d.shape[1]} cols (-{len(cols_d)} traidoras)')
print(f'   Caso D_strict: {df_automl_ds.shape[1]} cols (-{len(cols_ds)} traidoras)')

F3-M06: CASOS D y D_STRICT — ALERTA TEMPRANA REAL

📥 Caso C cargado: 33,621 × 36 cols
   Columnas: ['cred_matriculados_total', 'cred_superados_total', 'cred_titulacion', 'cred_superados_anio_medio', 'cred_superados_anio_1er', 'tasa_rendimiento', 'media_global', 'nota_1er_anio', 'nota_ultimo_anio', 'nota_acceso', 'rama', 'sexo', 'edad_entrada', 'pais_nombre', 'provincia', 'via_acceso', 'orden_preferencia', 'universidad_origen', 'tuvo_beca', 'n_anios_beca', 'situacion_laboral', 'nota_selectividad', 'pago_fraccionado', 'max_pagos', 'indicador_edad_inusual', 'indicador_interrupcion', 'indicador_sin_notas', 'anios_gap', 'tiene_gaps', 'ratio_avance', 'velocidad_creditos', 'mejora_notas', 'progreso_relativo', 'estabilidad_academica', 'anios_sin_beca', 'abandono']

--- CASO D (elimina 11 variables) ---
   Eliminadas: ['indicador_sin_notas', 'nota_ultimo_anio', 'cred_superados_total', 'cred_matriculados_total', 'cred_superados_anio_medio', 'tasa_rendimiento', 'ratio_avance', 'velocidad_creditos

In [3]:
# ============================================================================
# CELDA 3: EXPORTAR D + D_STRICT A data/automl/ (4 formatos)
# ============================================================================
# Solo exporta a data/automl/. El dataset para EDA (con texto legible)
# se genera en f3_m08_auditoria.ipynb como df_eda_final.
# ============================================================================

print(chr(10) + chr(45)*70)
print("EXPORTANDO CASOS D y D_STRICT A data/automl/")
print(chr(45)*70)

for caso, df_am, label in [
    ("D", df_automl_d, "Caso D"),
    ("D_strict", df_automl_ds, "Caso D_strict")]:

    print(f"\n--- {label} ---")
    for ext, func in [("parquet", lambda d,r: d.to_parquet(r, index=False)),
                       ("csv", lambda d,r: d.to_csv(r, sep=";", encoding="utf-8-sig", index=False, decimal=",")),
                       ("xlsx", lambda d,r: d.to_excel(r, index=False, sheet_name="datos")),
                       ("json", lambda d,r: d.to_json(r, orient="records", force_ascii=False))]:
        func(df_am, RUTA_AUTOML / f"df_exp_automl_{caso}.{ext}")
        print(f"  ✅ automl/df_exp_automl_{caso}.{ext}")

print(f"\n✅ Exportacion completada (solo a data/automl/)")



----------------------------------------------------------------------
EXPORTANDO CASOS D y D_STRICT A data/automl/
----------------------------------------------------------------------

--- Caso D ---
  ✅ automl/df_exp_automl_D.parquet


  ✅ automl/df_exp_automl_D.csv


  ✅ automl/df_exp_automl_D.xlsx
  ✅ automl/df_exp_automl_D.json

--- Caso D_strict ---
  ✅ automl/df_exp_automl_D_strict.parquet


  ✅ automl/df_exp_automl_D_strict.csv


  ✅ automl/df_exp_automl_D_strict.xlsx
  ✅ automl/df_exp_automl_D_strict.json

✅ Exportacion completada (solo a data/automl/)


In [4]:
# ============================================================================
# CELDA 4: QUICK BASELINES (D + D_strict)
# ============================================================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score, log_loss)
import time

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except:
    HAS_XGB = False

TARGET = 'abandono'
all_results = []

for caso, df_caso in [('D', df_automl_d), ('D_strict', df_automl_ds)]:
    print(f'\n{"="*70}')
    print(f'QUICK BASELINE — CASO {caso}')
    print(f'{"="*70}')

    X = df_caso.drop(columns=[TARGET])
    y = df_caso[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    print(f'Features: {X.shape[1]} | Train: {len(X_train):,} | Test: {len(X_test):,}')

    modelos = [
        ('DummyClassifier', DummyClassifier(strategy='stratified', random_state=42)),
        ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)),
        ('RandomForest', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
        ('GradientBoosting', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ]
    if HAS_XGB:
        modelos.append(('XGBoost', XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, eval_metric='logloss', verbosity=0)))

    for nombre, modelo in modelos:
        pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', modelo)])
        t0 = time.time()
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        try: y_prob = pipe.predict_proba(X_test)[:, 1]
        except: y_prob = np.zeros(len(y_test))
        t = time.time() - t0

        r = {'caso': caso, 'modelo': nombre,
             'accuracy': accuracy_score(y_test, y_pred),
             'balanced_acc': balanced_accuracy_score(y_test, y_pred),
             'precision': precision_score(y_test, y_pred, zero_division=0),
             'recall': recall_score(y_test, y_pred, zero_division=0),
             'f1': f1_score(y_test, y_pred, zero_division=0),
             'auc_roc': roc_auc_score(y_test, y_prob) if y_prob.sum() > 0 else 0,
             'mcc': matthews_corrcoef(y_test, y_pred),
             'tiempo': round(t, 2)}
        all_results.append(r)
        print(f'  {nombre:25s} AUC={r["auc_roc"]:.4f} | F1={r["f1"]:.4f} | Acc={r["accuracy"]:.4f} | MCC={r["mcc"]:.4f} | {t:.1f}s')

df_results = pd.DataFrame(all_results)

# Guardar
ruta_xlsx = RUTA_AUTOML / 'quick_baseline_casoD.xlsx'
with pd.ExcelWriter(ruta_xlsx, engine='openpyxl') as writer:
    df_results.to_excel(writer, sheet_name='todos', index=False)
    for caso in ['D', 'D_strict']:
        df_results[df_results['caso']==caso].sort_values('f1', ascending=False).to_excel(
            writer, sheet_name=f'caso_{caso}', index=False)
df_results.to_parquet(RUTA_AUTOML / 'quick_baseline_casoD.parquet', index=False)
print(f'\n💾 Guardado: {ruta_xlsx.name}')

for caso in ['D', 'D_strict']:
    mejor = df_results[df_results['caso']==caso].sort_values('f1', ascending=False).iloc[0]
    print(f'\n🏆 Mejor {caso}: {mejor["modelo"]} (AUC={mejor["auc_roc"]:.4f}, F1={mejor["f1"]:.4f})')


QUICK BASELINE — CASO D
Features: 24 | Train: 23,534 | Test: 10,087


  DummyClassifier           AUC=0.4908 | F1=0.2738 | Acc=0.5829 | MCC=-0.0186 | 0.2s


  LogisticRegression        AUC=0.9002 | F1=0.7328 | Acc=0.8518 | MCC=0.6325 | 2.7s


  RandomForest              AUC=0.9430 | F1=0.7977 | Acc=0.8899 | MCC=0.7264 | 1.2s


  GradientBoosting          AUC=0.9354 | F1=0.7733 | Acc=0.8764 | MCC=0.6924 | 3.9s


  XGBoost                   AUC=0.9433 | F1=0.8002 | Acc=0.8889 | MCC=0.7254 | 0.4s

QUICK BASELINE — CASO D_strict
Features: 22 | Train: 23,534 | Test: 10,087
  DummyClassifier           AUC=0.4908 | F1=0.2738 | Acc=0.5829 | MCC=-0.0186 | 0.1s


  LogisticRegression        AUC=0.8883 | F1=0.7206 | Acc=0.8464 | MCC=0.6178 | 2.9s


  RandomForest              AUC=0.9279 | F1=0.7758 | Acc=0.8793 | MCC=0.6990 | 0.6s


  GradientBoosting          AUC=0.9202 | F1=0.7589 | Acc=0.8701 | MCC=0.6756 | 3.8s


  XGBoost                   AUC=0.9288 | F1=0.7821 | Acc=0.8799 | MCC=0.7022 | 0.5s

💾 Guardado: quick_baseline_casoD.xlsx

🏆 Mejor D: XGBoost (AUC=0.9433, F1=0.8002)

🏆 Mejor D_strict: XGBoost (AUC=0.9288, F1=0.7821)


In [5]:
# ============================================================================
# CELDA 5: FEATURE IMPORTANCE (XGBoost, ambos casos)
# ============================================================================

if HAS_XGB:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    for caso, df_caso in [('D', df_automl_d), ('D_strict', df_automl_ds)]:
        X = df_caso.drop(columns=[TARGET])
        y = df_caso[TARGET]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

        xgb = Pipeline([('imputer', SimpleImputer(strategy='median')),
                        ('model', XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, eval_metric='logloss', verbosity=0))])
        xgb.fit(X_train, y_train)
        imp = pd.Series(xgb.named_steps['model'].feature_importances_, index=X.columns).sort_values(ascending=False)

        print(f'\n📊 Feature Importance — Caso {caso}:')
        for i, (feat, val) in enumerate(imp.head(10).items()):
            bar = '█' * int(val * 40)
            print(f'   {i+1:2d}. {feat:30s} {val:.4f} {bar}')

        fig, ax = plt.subplots(figsize=(10, 5))
        imp.head(15).plot.barh(ax=ax, color='#3182ce')
        ax.set_title(f'Feature Importance — Caso {caso}', fontweight='bold')
        ax.set_xlabel('Importancia')
        ax.invert_yaxis()
        plt.tight_layout()
        fig.savefig(str(RUTA_AUTOML / f'feature_importance_caso{caso}.png'), dpi=150, bbox_inches='tight')
        plt.close()
        print(f'   📊 Gráfico: feature_importance_caso{caso}.png')
else:
    print('XGBoost no disponible')


📊 Feature Importance — Caso D:
    1. estabilidad_academica          0.1656 ██████
    2. cred_superados_anio_1er        0.1227 ████
    3. situacion_laboral              0.1078 ████
    4. n_anios_beca                   0.0959 ███
    5. anios_sin_beca                 0.0603 ██
    6. nota_1er_anio                  0.0583 ██
    7. mejora_notas                   0.0529 ██
    8. tuvo_beca                      0.0307 █
    9. via_acceso                     0.0282 █
   10. pais_nombre                    0.0275 █


   📊 Gráfico: feature_importance_casoD.png



📊 Feature Importance — Caso D_strict:
    1. situacion_laboral              0.1713 ██████
    2. cred_superados_anio_1er        0.1678 ██████
    3. n_anios_beca                   0.1521 ██████
    4. anios_sin_beca                 0.1156 ████
    5. tuvo_beca                      0.0613 ██
    6. nota_1er_anio                  0.0323 █
    7. via_acceso                     0.0312 █
    8. indicador_interrupcion         0.0303 █
    9. edad_entrada                   0.0268 █
   10. pais_nombre                    0.0265 █


   📊 Gráfico: feature_importance_casoD_strict.png


In [6]:
# ============================================================================
# CELDA 6: HTML
# ============================================================================

nav_fases, nav_modulos = generar_html_navegacion_completa(fase_activa='fase3', modulo_activo='m06')

mejor_d = df_results[df_results['caso']=='D'].sort_values('f1', ascending=False).iloc[0]
mejor_ds = df_results[df_results['caso']=='D_strict'].sort_values('f1', ascending=False).iloc[0]

KPIS = [
    {'valor': fmt(len(df_automl_d)), 'titulo': 'Expedientes'},
    {'valor': f'{df_automl_d.shape[1]-1} / {df_automl_ds.shape[1]-1}', 'titulo': 'Features D/D_s'},
    {'valor': f'{mejor_d["auc_roc"]:.4f}', 'titulo': 'AUC (D)'},
    {'valor': f'{mejor_ds["auc_roc"]:.4f}', 'titulo': 'AUC (D_strict)'},
]

# Tabla variables eliminadas
s_intro = generar_seccion_html('🎯 Alerta Temprana: Casos D y D_strict', '''
<p>Datasets recomendados por el director del proyecto para <strong>predicción temprana real</strong>.</p>
<div style="display:grid;grid-template-columns:1fr 1fr;gap:15px;margin:15px 0;">
  <div style="background:#fff5f5;padding:15px;border-radius:10px;border-left:4px solid #e53e3e;">
    <h4 style="margin-top:0;color:#e53e3e;">Caso D (11 eliminadas)</h4>
    <p style="font-size:12px;font-family:monospace;">indicador_sin_notas, nota_ultimo_anio, cred_superados_total,
    cred_matriculados_total, cred_superados_anio_medio, tasa_rendimiento,
    ratio_avance, velocidad_creditos, progreso_relativo, media_global, cred_titulacion</p>
  </div>
  <div style="background:#faf5ff;padding:15px;border-radius:10px;border-left:4px solid #805ad5;">
    <h4 style="margin-top:0;color:#805ad5;">Caso D_strict (14 eliminadas)</h4>
    <p style="font-size:12px;font-family:monospace;">Todo lo de D + estabilidad_academica, mejora_notas</p>
  </div>
</div>''')

# Tablas de resultados
contenido = ''
for caso in ['D', 'D_strict']:
    df_c = df_results[df_results['caso']==caso].sort_values('f1', ascending=False)
    tabla = '<table style="width:100%;border-collapse:collapse;">\n'
    color_h = '#e53e3e' if caso == 'D' else '#805ad5'
    tabla += f'<tr style="background:{color_h};color:white;">'
    for h in ['Modelo','Acc','Bal.Acc','F1','AUC','MCC','Tiempo']: tabla += f'<th style="padding:8px;text-align:center;">{h}</th>'
    tabla += '</tr>\n'
    for i, (_, row) in enumerate(df_c.iterrows()):
        bg = '#f7fafc' if i%2 else 'white'
        tabla += f'<tr style="background:{bg};"><td style="padding:6px;">{row["modelo"]}</td>'
        for c in ['accuracy','balanced_acc','f1','auc_roc','mcc']:
            v=row[c]; col='#38a169' if v>0.7 else '#ed8936' if v>0.5 else '#e53e3e'
            tabla += f'<td style="text-align:center;color:{col};">{v:.4f}</td>'
        tabla += f'<td style="text-align:center;">{row["tiempo"]:.1f}s</td></tr>\n'
    tabla += '</table>'

    # Feature importance image
    img = ''
    img_path = RUTA_AUTOML / f'feature_importance_caso{caso}.png'
    if img_path.exists():
        import base64
        with open(img_path, 'rb') as f: img_b64 = base64.b64encode(f.read()).decode()
        img = f'<img src="data:image/png;base64,{img_b64}" style="max-width:100%;border-radius:8px;margin-top:15px;">'

    mejor = df_c.iloc[0]
    contenido += generar_seccion_html(f'📊 Caso {caso} ({df_automl_d.shape[1]-1 if caso=="D" else df_automl_ds.shape[1]-1} features)',
        f'<p><strong>Mejor:</strong> {mejor["modelo"]} (AUC={mejor["auc_roc"]:.4f}, F1={mejor["f1"]:.4f})</p>\n{tabla}\n{img}')

s_concl = generar_seccion_html('✅ Conclusión', f'''
<div style="background:#f0fff4;padding:20px;border-radius:10px;border-left:4px solid #38a169;">
  <p><strong>Caso D:</strong> {mejor_d["modelo"]} — AUC = {mejor_d["auc_roc"]:.4f}, F1 = {mejor_d["f1"]:.4f}</p>
  <p><strong>Caso D_strict:</strong> {mejor_ds["modelo"]} — AUC = {mejor_ds["auc_roc"]:.4f}, F1 = {mejor_ds["f1"]:.4f}</p>
  <p>Estas métricas son <strong>realistas</strong>. El modelo aprende patrones genuinos de abandono
  (vía de acceso, nota primer año, becas...) en lugar de leer la respuesta escondida.</p>
  <p>💡 Un AUC de 0.75-0.85 con variables de alerta temprana es un <strong>ÉXITO REAL</strong>.</p>
</div>''')

html = render_base_html(titulo='M06: Alerta Temprana (D + D_strict)', icono='🚨',
    subtitulo='Fase 3: Feature Engineering',
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=generar_kpis_html(KPIS) + s_intro + contenido + s_concl,
    notebook_nombre='f3_m06_alerta_temprana.ipynb', notebook_carpeta='fase3_features')
ruta_html = RUTA_FASE3_HTML / 'm06_alerta_temprana.html'
guardar_html(html, ruta_html)
print(f'\n✅ HTML: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m06_alerta_temprana.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m06_alerta_temprana.html


In [7]:
# ============================================================================
# CELDA 7: RESUMEN FINAL
# ============================================================================

mejor_d = df_results[df_results['caso']=='D'].sort_values('f1', ascending=False).iloc[0]
mejor_ds = df_results[df_results['caso']=='D_strict'].sort_values('f1', ascending=False).iloc[0]

print('\n' + '='*70)
print('✅ F3-M06 COMPLETADO — ALERTA TEMPRANA (D + D_STRICT)')
print('='*70)

print(f'\n📊 Caso D:        {df_automl_d.shape[1]} cols ({df_automl_d.shape[1]-1} features)')
print(f'   🏆 Mejor: {mejor_d["modelo"]} (AUC={mejor_d["auc_roc"]:.4f}, F1={mejor_d["f1"]:.4f})')

print(f'\n📊 Caso D_strict: {df_automl_ds.shape[1]} cols ({df_automl_ds.shape[1]-1} features)')
print(f'   🏆 Mejor: {mejor_ds["modelo"]} (AUC={mejor_ds["auc_roc"]:.4f}, F1={mejor_ds["f1"]:.4f})')

print(f'\n📁 Archivos generados:')
for caso in ['D', 'D_strict']:
    print(f'   automl/df_exp_automl_{caso}.parquet/csv/xlsx/json')

print(f'   automl/quick_baseline_casoD.xlsx (ambos casos)')
print(f'   automl/feature_importance_casoD.png')
print(f'   automl/feature_importance_casoD_strict.png')
print(f'   HTML: {ruta_html}')

print(f'\n💡 Un AUC de ~0.75-0.85 con alerta temprana es un ÉXITO REAL.')
print(f'   El modelo aprende patrones genuinos, no lee la respuesta.')
print('='*70)


✅ F3-M06 COMPLETADO — ALERTA TEMPRANA (D + D_STRICT)

📊 Caso D:        25 cols (24 features)
   🏆 Mejor: XGBoost (AUC=0.9433, F1=0.8002)

📊 Caso D_strict: 23 cols (22 features)
   🏆 Mejor: XGBoost (AUC=0.9288, F1=0.7821)

📁 Archivos generados:
   automl/df_exp_automl_D.parquet/csv/xlsx/json
   automl/df_exp_automl_D_strict.parquet/csv/xlsx/json
   automl/quick_baseline_casoD.xlsx (ambos casos)
   automl/feature_importance_casoD.png
   automl/feature_importance_casoD_strict.png
   HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m06_alerta_temprana.html

💡 Un AUC de ~0.75-0.85 con alerta temprana es un ÉXITO REAL.
   El modelo aprende patrones genuinos, no lee la respuesta.
